In [1]:
pip install scikit-learn gensim konlpy pandas matplotlib seaborn pyldavis

Defaulting to user installation because normal site-packages is not writeable
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp313-cp313-win_amd64.whl.metadata (6.4 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---- ----------------------------------- 2.9/24.4 MB 19.1 MB/s eta 0:00:02
   ------------ --------------------------- 7.3/24.4 MB 19.9 MB/s eta 0:00:01
   ------------------ --------------------- 11.5/24.4 MB 19.7 MB/s eta 0:00:01
   ------------------------ --------------- 14.7/24.4 MB 18.4 MB/s eta 0:00:01
   -------------------------- ------------- 16.3/24.4 MB 17.6 MB/s eta 0:00:01
   ---------------------------------- ----- 21.0/24.4 MB 17.3 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.4 MB 17.1 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 16.4 MB/s  0:00:01
   -------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\WINDOWS 11\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\WINDOWS 11\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [4]:
from konlpy.tag import Okt

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [7]:
INPUT_CSV = "C:\\Users\\WINDOWS 11\\Desktop\\kpop_agenda\\data\\top300_filtered.csv"  # ⚠️ CHANGE THIS
ARTICLES_DIR = "C:/Users/Мансур/OneDrive/Documents/GitHub/k-pop-agenda/data"  # ⚠️ CHANGE THIS
OUTPUT_CSV = "lda_topic_results.csv"

# LDA Parameters
N_TOPICS = 5  # Number of topics (adjust based on your needs)
RANDOM_STATE = 42

# Korean stopwords
KOREAN_STOPWORDS = [
    '이', '그', '저', '것', '수', '등', '들', '및', '때문', '위해',
    '통해', '대해', '관련', '경우', '따르', '위', '중', '더', '가장',
    '또', '또한', '그리고', '하지만', '그러나', '있다', '없다', '되다',
    '하다', '이다', '아니다', '것이다', '수있다', '있는', '없는',
    '기자', '뉴스', '보도', '연합뉴스', '스포츠', '엔터'
]

In [8]:
# Now initialize the tokenizer
try:
    from konlpy.tag import Mecab
    tokenizer = Mecab()
    print("Using Mecab tokenizer (faster)")
except:
    tokenizer = Okt()
    print("Using Okt tokenizer")

def preprocess_korean_text(text):
    """Preprocess Korean text for LDA"""
    if not text or pd.isna(text):
        return ""
    
    text = str(text)
    
    # Extract nouns (best for topic modeling)
    words = tokenizer.nouns(text)
    
    # Filter stopwords and short words
    words = [w for w in words if w not in KOREAN_STOPWORDS and len(w) > 1]
    
    return ' '.join(words)

Using Okt tokenizer


In [9]:
def load_and_preprocess_documents(csv_path, articles_dir):
    """Load articles from CSV and text files"""
    print("Loading data...")
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} articles from CSV")
    
    documents = []
    processed_docs = []
    valid_indices = []
    
    print("\nReading and preprocessing article files...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Adjust based on your CSV column name
        article_id = row.get('article_id', idx)
        file_path = Path(articles_dir) / f"{article_id}.txt"
        
        if file_path.exists():
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read().strip()
                
                if text:
                    documents.append(text)
                    processed_text = preprocess_korean_text(text)
                    processed_docs.append(processed_text)
                    valid_indices.append(idx)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    
    df = df.iloc[valid_indices].reset_index(drop=True)
    
    print(f"\n✅ Successfully loaded {len(documents)} articles")
    print(f"Sample processed text: {processed_docs[0][:100]}...")
    
    return df, documents, processed_docs

# Load data
df, raw_documents, processed_documents = load_and_preprocess_documents(INPUT_CSV, ARTICLES_DIR)

Loading data...
Loaded 300 articles from CSV

Reading and preprocessing article files...


100%|██████████| 300/300 [00:00<00:00, 5727.21it/s]



✅ Successfully loaded 0 articles


IndexError: list index out of range

In [ ]:
print("\nCreating document-term matrix...")
vectorizer = CountVectorizer(
    max_df=0.8,  # Ignore words in >80% of documents
    min_df=2,    # Ignore words in <2 documents
    max_features=1000
)
doc_term_matrix = vectorizer.fit_transform(processed_documents)
feature_names = vectorizer.get_feature_names_out()

print(f"✅ Document-term matrix shape: {doc_term_matrix.shape}")
print(f"   Documents: {doc_term_matrix.shape[0]}")
print(f"   Vocabulary size: {doc_term_matrix.shape[1]}")

In [ ]:
print("\n" + "="*60)
print("TRAINING LDA MODEL")
print("="*60)

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    max_iter=20,
    learning_method='online',
    n_jobs=-1,
    verbose=1
)

lda_output = lda_model.fit_transform(doc_term_matrix)

print(f"   LDA training completed!")
print(f"   Perplexity: {lda_model.perplexity(doc_term_matrix):.2f}")
print(f"   Log-likelihood: {lda_model.score(doc_term_matrix):.2f}")

In [ ]:
def display_topics(model, feature_names, n_top_words=15):
    """Display top words for each topic"""
    topics = []
    print("\n" + "="*60)
    print("TOP WORDS PER TOPIC")
    print("="*60)
    
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
        
        print(f"\nTopic {topic_idx}:")
        print(f"  {', '.join(top_words)}")
    
    return topics

topics = display_topics(lda_model, feature_names)

In [ ]:
dominant_topics = lda_output.argmax(axis=1)
topic_probs = lda_output.max(axis=1)

df['lda_topic'] = dominant_topics
df['lda_topic_prob'] = topic_probs

# Count documents per topic
print("\n" + "="*60)
print("TOPIC DISTRIBUTION")
print("="*60)
topic_counts = df['lda_topic'].value_counts().sort_index()

print(f"{'Topic ID':<10} | {'Article Count':<15} | {'Percentage'}")
print("-"*50)
for topic_id, count in topic_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{topic_id:<10} | {count:<15} | {percentage:.1f}%")

print(f"\nTotal articles: {len(df)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

topic_counts.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Articles per Topic', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Topic ID', fontsize=12)
axes[0].set_ylabel('Number of Articles', fontsize=12)
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(topic_counts, labels=[f'Topic {i}' for i in topic_counts.index], 
            autopct='%1.1f%%', startangle=90, colors=sns.color_palette('husl', len(topic_counts)))
axes[1].set_title('Topic Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('lda_topic_distribution.png', dpi=300, bbox_inches='tight')
plt.show()